In [ ]:
import locale
locale.getpreferredencoding = lambda: 'UTF-8'

In [ ]:
!pip install git+https://github.com/openai/whisper.git
!pip install torch
!pip install pydub
!pip install openai
!pip install pyannote.audio
!pip install langdetect
!pip install ffmpeg
!pip install --no-cache-dir --force-reinstall langdetect
!pip install azure-cognitiveservices-speech

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-zbbvr7_l
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-zbbvr7_l
  Resolved https://github.com/openai/whisper.git to commit 517a43ecd132a2089d85f4ebc044728a71d49f6e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.4 MB/s eta 0:00:00
   ━━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 49.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=0d65a75bab0731a38eb19799280883054d579c30002975a0d7036fa5c6937329
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
  Preparing metadata (setup.py) ... done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6082 sha256=a99b467c5c7c9b6085adaa8d772d2ab6ea5c01487243b27f53d701475899cb7f
  Stored in directory: /root/.cache/pip/wheels/56/30/c5/576bdd729f3bc062d62a551be7fefd6ed2f761901568171e4e
Successfully built ffmpeg
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 94.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=a0218c73fe1e378e2162d5224ac8b8cf02b347e24

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 52.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
!git clone https://liz3-github/WBW_Run_Model.git

fatal: destination path 'WBW_Run_Model' already exists and is not an empty directory.


In [1]:
import whisper
from pydub import AudioSegment
import os
import json
import math
from tqdm import tqdm
import requests
import subprocess
import time
import logging

# Reprocess all files if True
REPROCESS = True

os.chdir("/content")
print(f"Current working directory: {os.getcwd()}")

# Configure logging to a file
logging.basicConfig(
    filename='audio_processing.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

drive.mount('/content/drive', force_remount=True)

# GitHub credentials (token) & repo URL for fetching scripts
GITHUB_TOKEN = ""
GITHUB_API_URL = ""
headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3.raw"
}

# Input audio folder and output base path
AUDIO_FOLDER = ""
OUTPUT_BASE = ""
PROGRESS_FILE = os.path.join(OUTPUT_BASE, "progress.json")

print(f"AUDIO_FOLDER path: {AUDIO_FOLDER}")
print(f"OUTPUT_BASE path: {OUTPUT_BASE}")
print(f"PROGRESS_FILE path: {PROGRESS_FILE}")

# List of method scripts (in your repo) to run on each audio
methods = [
    "Method1/1_Prompt_samll_WBW.py",
    "Method1/1_Prompt_medium_WBW.py",
    "Method2/2_Langdetect_small_WBW.py",
    "Method2/2_Langdetect_medium_WBW.py",
    "Method3/3_overlap30s_small.py",
    "Method3/3_overlap30s_medium.py",
    "Method4/4_Detect_first30s_small_WBW.py",
    "Method4/4_Detect_first30s_medium_WBW.py",
    "Method5/5_PY+WH_small_WBW.py",
    "Method5/5_PY+WH_medium_WBW.py",
    "Method6/6_Azure+PY.py",





]

def get_github_script(script_path, input_file, output_folder):
    """
    Fetch the script from GitHub and replace placeholders with real paths.
    """
    url = GITHUB_API_URL + script_path
    print(f"Fetching script from: {url}")

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        script_content = response.text

        # Remove Colab import if present
        script_content = script_content.replace("from google.colab import files", "")

        # Replace placeholders with actual file/folder
        script_content = script_content.replace(
            'AUDIO_FILE = "{input_file}"',
            f'AUDIO_FILE = "{input_file}"'
        )


        script_content = script_content.replace(
            'OUTPUT_FOLDER = "{output_folder}"',
            f'OUTPUT_FOLDER = "{output_folder}"'
        )

        # Adjust main() signature if needed
        script_content = script_content.replace(
            "def main():",
            "def main(input_file, output_file):"
        )
        script_content = script_content.replace(
            "main()",
            f"main('{input_file}', '{output_folder}')"
        )

        return script_content

    except requests.RequestException as e:
        error_message = f"Failed to fetch script from GitHub: {url}. Error: {str(e)}"
        print(error_message)
        logging.error(error_message)
        raise

def process_audio(audio_file, total_files, file_index, progress_bar):
    """
    Run each method script on this audio file.
    """
    if not os.path.exists(audio_file):
        print(f"Audio file not found: {audio_file}")
        return False

    audio_name = os.path.splitext(os.path.basename(audio_file))[0]
    video_folder = os.path.join(OUTPUT_BASE, audio_name)

    for method_index, method in enumerate(methods, 1):
        method_name = method.split('/')[0]
        method_folder = os.path.join(video_folder, method_name)
        os.makedirs(method_folder, exist_ok=True)

        # Example output file name (often unused by the actual script)
        output_file = os.path.join(
            method_folder,
            f"{os.path.splitext(os.path.basename(method))[0]}_ignored.json"
        )

        progress_message = f"Processing file {file_index}/{total_files}, method {method_index}/{len(methods)}: {audio_name} - {method}"
        print(progress_message)
        logging.info(progress_message)

        try:
            script_content = get_github_script(method, audio_file, method_folder)

            temp_script = f"/tmp/{os.path.basename(method)}"

            # Save modified script locally
            with open(temp_script, 'w') as f:
                f.write(script_content)

            # Run the script
            command = f"python3 '{temp_script}' --input '{audio_file}' --output '{output_file}'"
            print(f"Executing command: {command}")
            result = subprocess.run(command, shell=True, check=True, capture_output=True, text=True)
            print(result.stdout)

            # Update progress bar
            progress_bar.set_description(f"Processing {os.path.basename(audio_file)} - {method}")
            progress_bar.update(1)

            logging.info(f"Successfully processed {audio_file} with {method}")

            # Remove temporary script
            os.remove(temp_script)
        except Exception as e:
            error_message = f"Error processing {audio_file} with {method}: {str(e)}"
            print(error_message)
            logging.error(error_message)

    return True

def process_batch(batch, start_index, total_files):
    """
    Process a batch of audio files, showing a progress bar.
    """
    with tqdm(total=len(batch) * len(methods), desc="Overall Progress") as progress_bar:
        for file_index, audio_file in enumerate(batch, start_index):
            full_path = os.path.join(AUDIO_FOLDER, audio_file)
            print(f"\nStarting to process file {file_index}/{total_files}: {audio_file}")
            logging.info(f"Starting to process file {file_index}/{total_files}: {audio_file}")

            success = process_audio(full_path, total_files, file_index, progress_bar)

            if success:
                update_progress(audio_file)

            print(f"Finished processing file {file_index}/{total_files}: {audio_file}")
            logging.info(f"Finished processing file {file_index}/{total_files}: {audio_file}")

    time.sleep(10)

def update_progress(processed_file):
    """
    Append the processed file to progress.json so we skip it next time.
    """
    try:
        with open(PROGRESS_FILE, 'r+') as f:
            progress = json.load(f)
            progress['processed_files'].append(processed_file)
            progress['total_processed'] = len(progress['processed_files'])
            f.seek(0)
            json.dump(progress, f, indent=2)
            f.truncate()
    except IOError as e:
        print(f"Error updating progress file: {e}")
        logging.error(f"Error updating progress file: {e}")

def main():
    """
    Main entry: list all .wav files, handle progress, batch process, etc.
    """
    print("Starting main function")

    print(f"Audio folder contents: {os.listdir(AUDIO_FOLDER)}")

    audio_files = [f for f in os.listdir(AUDIO_FOLDER) if f.endswith('.wav')]
    total_files = len(audio_files)
    print(f"Total audio files found: {total_files}")

    # Reset progress if reprocessing
    if REPROCESS:
        print("Reprocessing all files, ignoring previous progress.")
        if os.path.exists(PROGRESS_FILE):
            os.remove(PROGRESS_FILE)

    # Create progress file if not exist
    if not os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'w') as f:
            json.dump({'processed_files': [], 'total_processed': 0}, f, indent=2)

    with open(PROGRESS_FILE, 'r') as f:
        progress = json.load(f)

    # Skip files already processed if not reprocessing
    if not REPROCESS:
        audio_files = [f for f in audio_files if f not in progress['processed_files']]

    print(f"Files to be processed: {len(audio_files)}")
    if not audio_files:
        print("No new files to process. Set REPROCESS = True to reprocess all files.")
        return

    # Process in small groups to avoid overload
    batch_size = 5
    for i in range(0, len(audio_files), batch_size):
        batch = audio_files[i:i+batch_size]
        process_batch(batch, i+1, total_files)

    print("Finished processing all audio files.")
    print(f"Total {len(audio_files)} audio files have been processed. Check audio_processing.log for details.")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"An error occurred in main: {e}")
        logging.error(f"An error occurred in main: {e}")



ModuleNotFoundError: No module named 'whisper'